# Distillation: On-Policy with Teacher KL

This notebook demonstrates on-policy distillation where the student model generates samples and learns by minimizing KL divergence to a teacher model.

**Goal**: Train a student model to match a teacher's behavior on new samples.

**Method**: Student generates responses, then learns by minimizing KL(student || teacher).

In [ ]:
import os
os.environ['TINKER_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

## On-Policy Distillation Explained

### Process
1. **Student samples**: Generate responses from current student model
2. **Teacher evaluates**: Compute teacher's log probabilities on student samples
3. **KL penalty**: Minimize divergence between student and teacher
4. **Update**: Gradient descent to reduce KL

### Why On-Policy?
- **Adaptive**: Teaches student exactly where it struggles
- **Distribution match**: Student learns on its own distribution
- **Better generalization**: Avoids distribution shift from off-policy data

In [ ]:
# Training configuration
config = {
    'model_name': 'meta-llama/Llama-3.2-1B',
    'dataset': 'tulu3',          # Prompts from Tulu3
    'learning_rate': 1e-4,
    'groups_per_batch': 32,
    'lora_rank': 32,
}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## Run Training

In [ ]:
!python -m tinker_cookbook.recipes.distillation.on_policy_distillation \
    model_name="meta-llama/Llama-3.2-1B" \
    dataset=tulu3 \
    learning_rate=1e-4 \
    groups_per_batch=32 \
    lora_rank=32

## Understanding the Output

### Key Metrics
| Metric | Description |
|--------|-------------|
| `teacher_kl` | KL divergence to teacher (should decrease) |
| `optim/entropy` | Student's output entropy |
| `time/sample` | Time to generate samples |
| `time/compute_kl_penalty` | Time for KL computation |

### Sample Output
The model shows generated samples with teacher KL being computed:
```
Starting compute_kl_penalty
compute_kl_penalty took X seconds
```

## Expected Learning Curve

```
Step  0: teacher_kl=3.5
Step 10: teacher_kl=2.5
Step 50: teacher_kl=1.5
Step 100: teacher_kl=1.0
```

The KL divergence decreases as the student learns to match the teacher.

## DeepMath Configuration

For math reasoning distillation (achieves ~65% on AIME'24):

In [ ]:
# DeepMath distillation config
deepmath_config = {
    'model_name': 'Qwen/Qwen3-8B-Base',
    'dataset': 'deepmath',
    'learning_rate': 1e-4,
    'groups_per_batch': 512,
    'lora_rank': 128,
}

print("DeepMath config:")
for k, v in deepmath_config.items():
    print(f"  {k}: {v}")

## Analyze Results

In [ ]:
import json
import glob

# Find the latest metrics file
metrics_files = glob.glob('/home/ubuntu/tinker-examples/distillation/distill-*/metrics.jsonl')
if metrics_files:
    latest = max(metrics_files)
    print(f"Reading: {latest}\n")
    
    kl_history = []
    
    with open(latest) as f:
        for line in f:
            data = json.loads(line)
            if 'teacher_kl' in data:
                kl_history.append(data['teacher_kl'])
    
    if kl_history:
        print(f"Starting KL: {kl_history[0]:.4f}")
        print(f"Current KL: {kl_history[-1]:.4f}")
        print(f"Reduction: {kl_history[0] - kl_history[-1]:.4f}")